In [4]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.models.model_fno import BaselineCNN
from src.data.data_processing import collect_stats, get_train_val_files, normalize

time = ['1 days', '2 days', '4 days', '7 days', '11 days', '17 days',
        '25 days', '37 days', '53 days', '77 days', '111 days', '158 days',
        '226 days', '323 days', '1.3 years', '1.8 years', '2.6 years',
        '3.6 years', '5.2 years', '7.3 years', '10.4 years', '14.8 years',
        '21.1 years', '30.0 years']

plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["image.cmap"] = "jet"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [5]:
def r2(x, y):
    x = x.flatten()
    y = y.flatten()
    zx = (x - np.mean(x)) / np.std(x, ddof=1)
    zy = (y - np.mean(y)) / np.std(y, ddof=1)
    r = np.sum(zx * zy) / (len(x) - 1)
    return r ** 2

def MAE(x, y):
    x = x.flatten()
    y = y.flatten()
    return np.mean(np.abs(x - y))

def build_input(data, stats):
    porosity = data["porosity"].astype(np.float32)
    perm_r = np.log10(data["perm_r"].astype(np.float32) + 1e-12)
    perm_z = np.log10(data["perm_z"].astype(np.float32) + 1e-12)

    nz, nx = porosity.shape

    inj_rate = np.full((nz, nx), data["inj_rate"], dtype=np.float32)
    temperature = np.full((nz, nx), data["temperature"], dtype=np.float32)
    depth = np.full((nz, nx), data["depth"], dtype=np.float32)
    swi = np.full((nz, nx), data["Swi"], dtype=np.float32)
    lam = np.full((nz, nx), data["lam"], dtype=np.float32)

    perf_mask = np.zeros((nz, nx), dtype=np.float32)
    z0, z1 = int(data["perf_interval"][0]), int(data["perf_interval"][1])
    perf_mask[z0:z1 + 1, 0] = 1.0

    porosity = normalize(porosity, stats["porosity"]["mean"], stats["porosity"]["std"])
    perm_r = normalize(perm_r, stats["perm_r"]["mean"], stats["perm_r"]["std"])
    perm_z = normalize(perm_z, stats["perm_z"]["mean"], stats["perm_z"]["std"])
    inj_rate = normalize(inj_rate, stats["inj_rate"]["mean"], stats["inj_rate"]["std"])
    temperature = normalize(temperature, stats["temperature"]["mean"], stats["temperature"]["std"])
    depth = normalize(depth, stats["depth"]["mean"], stats["depth"]["std"])
    swi = normalize(swi, stats["Swi"]["mean"], stats["Swi"]["std"])
    lam = normalize(lam, stats["lam"]["mean"], stats["lam"]["std"])

    x = np.stack(
        [
            porosity,
            perm_r,
            perm_z,
            inj_rate,
            temperature,
            depth,
            swi,
            lam,
            perf_mask,
        ],
        axis=0,
    )

    return x

def pcolor_factory(nz):
    dx = np.cumsum(3.5938 * np.power(1.035012, range(200))) + 0.1
    X, Y = np.meshgrid(dx, np.linspace(0, 200, num=nz))

    def pcolor(x):
        plt.jet()
        plt.xlim([0, 4000])
        return plt.pcolor(X, Y, np.flipud(x))

    return pcolor

In [10]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
os.chdir(project_root)
sys.path.append(str(project_root))

print("Working directory is now:", Path.cwd())

Working directory is now: c:\Users\KirstyLocal\Projects\co2-plume-operator-learning


In [11]:
# Keep these settings aligned with the training run you want to evaluate
checkpoint_path = "checkpoints/best_baseline_phase1c.pt"

all_train_files, all_val_files = get_train_val_files()

train_files = all_train_files[:200]
val_files = all_val_files[:50]

stats = collect_stats(train_files, max_files=200)

model = BaselineCNN().to(device)
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

print("Loaded checkpoint:", checkpoint_path)
print("Train files used for stats:", len(train_files))
print("Validation files to inspect:", len(val_files))

RuntimeError: Error(s) in loading state_dict for BaselineCNN:
	Missing key(s) in state_dict: "encoder.0.weight", "encoder.0.bias", "encoder.2.weight", "encoder.2.bias", "encoder.4.weight", "encoder.4.bias", "encoder.6.weight", "encoder.6.bias", "gas_head.0.weight", "gas_head.0.bias", "gas_head.2.weight", "gas_head.2.bias", "pressure_head.0.weight", "pressure_head.0.bias", "pressure_head.2.weight", "pressure_head.2.bias". 
	Unexpected key(s) in state_dict: "net.0.weight", "net.0.bias", "net.2.weight", "net.2.bias", "net.4.weight", "net.4.bias", "net.6.weight", "net.6.bias", "net.8.weight", "net.8.bias". 

In [ ]:
# Evaluate the checkpoint on the validation subset
gas_saturation_r2, pressure_buildup_r2 = [], []
gas_saturation_MAE, pressure_buildup_MAE = [], []

for file_path in val_files:
    with np.load(file_path) as data:
        gas_saturation = data["gas_saturation"]
        pressure_buildup = data["pressure_buildup"]

        x = build_input(data, stats)
        x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            gas_pred, pressure_pred = model(x_tensor)

        gas_pred = gas_pred.squeeze(0).cpu().numpy()
        pressure_pred = pressure_pred.squeeze(0).cpu().numpy()

        pressure_pred = pressure_pred * stats["pressure"]["std"] + stats["pressure"]["mean"]

        gas_saturation_pred = np.transpose(gas_pred, (1, 2, 0))
        pressure_buildup_pred = np.transpose(pressure_pred, (1, 2, 0))

        gas_saturation_r2.append(r2(gas_saturation, gas_saturation_pred))
        pressure_buildup_r2.append(r2(pressure_buildup, pressure_buildup_pred))
        gas_saturation_MAE.append(MAE(gas_saturation, gas_saturation_pred))
        pressure_buildup_MAE.append(MAE(pressure_buildup, pressure_buildup_pred))

print("--------------")
print(f"Average validation set gas saturation R2 score is: {np.mean(gas_saturation_r2):.6f}")
print(f"Average validation set pressure buildup R2 score is: {np.mean(pressure_buildup_r2):.6f}")
print("--------------")
print(f"Average validation set gas saturation MAE is: {np.mean(gas_saturation_MAE):.6f}")
print(f"Average validation set pressure buildup MAE is: {np.mean(pressure_buildup_MAE):.6f}")

In [ ]:
# Inspect one validation sample in detail
sample_idx = 0
sample_path = val_files[sample_idx]

with np.load(sample_path) as data:
    gas_true = data["gas_saturation"]
    pressure_true = data["pressure_buildup"]

    x = build_input(data, stats)
    x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        gas_pred, pressure_pred = model(x_tensor)

    gas_pred = gas_pred.squeeze(0).cpu().numpy()
    pressure_pred = pressure_pred.squeeze(0).cpu().numpy()
    pressure_pred = pressure_pred * stats["pressure"]["std"] + stats["pressure"]["mean"]

    gas_pred = np.transpose(gas_pred, (1, 2, 0))
    pressure_pred = np.transpose(pressure_pred, (1, 2, 0))

nz = gas_true.shape[0]
pcolor = pcolor_factory(nz)

print("Sample path:", sample_path)
print("Gas R2:", r2(gas_true, gas_pred))
print("Pressure R2:", r2(pressure_true, pressure_pred))
print("Gas MAE:", MAE(gas_true, gas_pred))
print("Pressure MAE:", MAE(pressure_true, pressure_pred))

In [ ]:
# Starter-style final-time comparison plots
plt.figure(figsize=(15, 12))

plt.subplot(3, 2, 1)
pcolor(gas_true[:, :, -1])
plt.colorbar()
plt.title(f"true gas_saturation {time[-1]}")

plt.subplot(3, 2, 2)
pcolor(gas_pred[:, :, -1])
plt.colorbar()
plt.title(f"predicted gas_saturation {time[-1]}")

plt.subplot(3, 2, 3)
pcolor(np.abs(gas_true[:, :, -1] - gas_pred[:, :, -1]))
plt.colorbar()
plt.title(f"gas absolute error {time[-1]}")

plt.subplot(3, 2, 4)
pcolor(pressure_true[:, :, -1])
plt.colorbar()
plt.title(f"true pressure_buildup {time[-1]}")

plt.subplot(3, 2, 5)
pcolor(pressure_pred[:, :, -1])
plt.colorbar()
plt.title(f"predicted pressure_buildup {time[-1]}")

plt.subplot(3, 2, 6)
pcolor(np.abs(pressure_true[:, :, -1] - pressure_pred[:, :, -1]))
plt.colorbar()
plt.title(f"pressure absolute error {time[-1]}")

plt.tight_layout()
plt.show()

In [ ]:
# Compare truth vs prediction at multiple times for gas saturation
selected_times = [0, 7, 15, 23]

fig, axes = plt.subplots(len(selected_times), 3, figsize=(14, 14))

for row, t_idx in enumerate(selected_times):
    plt.sca(axes[row, 0])
    pcolor(gas_true[:, :, t_idx])
    plt.colorbar(ax=axes[row, 0])
    axes[row, 0].set_title(f"true gas {time[t_idx]}")

    plt.sca(axes[row, 1])
    pcolor(gas_pred[:, :, t_idx])
    plt.colorbar(ax=axes[row, 1])
    axes[row, 1].set_title(f"predicted gas {time[t_idx]}")

    plt.sca(axes[row, 2])
    pcolor(np.abs(gas_true[:, :, t_idx] - gas_pred[:, :, t_idx]))
    plt.colorbar(ax=axes[row, 2])
    axes[row, 2].set_title(f"gas abs error {time[t_idx]}")

plt.tight_layout()
plt.show()

In [ ]:
# Compare truth vs prediction at multiple times for pressure buildup
selected_times = [0, 7, 15, 23]

fig, axes = plt.subplots(len(selected_times), 3, figsize=(14, 14))

for row, t_idx in enumerate(selected_times):
    plt.sca(axes[row, 0])
    pcolor(pressure_true[:, :, t_idx])
    plt.colorbar(ax=axes[row, 0])
    axes[row, 0].set_title(f"true pressure {time[t_idx]}")

    plt.sca(axes[row, 1])
    pcolor(pressure_pred[:, :, t_idx])
    plt.colorbar(ax=axes[row, 1])
    axes[row, 1].set_title(f"predicted pressure {time[t_idx]}")

    plt.sca(axes[row, 2])
    pcolor(np.abs(pressure_true[:, :, t_idx] - pressure_pred[:, :, t_idx]))
    plt.colorbar(ax=axes[row, 2])
    axes[row, 2].set_title(f"pressure abs error {time[t_idx]}")

plt.tight_layout()
plt.show()